In [1]:
# import os
# import re
# import shutil
# from urllib.parse import urljoin
# import zipfile
# import requests
# from bs4 import BeautifulSoup
# from tqdm import tqdm

# # ---------------------------------------------------------
# # Configuration
# # ---------------------------------------------------------
# BASE_URL = (
#     "https://www.cms.gov/data-research/statistics-trends-and-reports/"
#     "medicare-claims-synthetic-public-use-files/"
#     "cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-{}"
# )

# # Root directory in Kaggle
# ROOT_DIR = "/kaggle/working/synpuf_data"

# # Choose which samples to process (e.g., range(1, 4) for testing or range(1, 21) for all)
# SAMPLE_RANGE = range(1, 21)

# # Target file patterns: Inpatient, Outpatient, and Beneficiary Summaries (2008-2010)
# TARGET_PATTERNS = [
#     re.compile(r"inpatient_claims", re.IGNORECASE),
#     re.compile(r"outpatient_claims", re.IGNORECASE),
#     re.compile(r"(2008|2009|2010)_beneficiary_summary", re.IGNORECASE),
# ]

# HEADERS = {
#     "User-Agent": (
#         "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#         "AppleWebKit/537.36 (KHTML, like Gecko) "
#         "Chrome/124.0.0.0 Safari/537.36"
#     )
# }


# def is_target_file(url: str) -> bool:
#     """Checks if the URL points to one of the requested target files."""
#     url_lower = url.lower()
#     # Explicitly exclude carrier and prescription drug files
#     if "carrier" in url_lower or "prescription" in url_lower:
#         return False
#     return any(pattern.search(url_lower) for pattern in TARGET_PATTERNS)


# def download_and_extract(url: str, dest_folder: str):
#     """Downloads a zip file, extracts CSVs into dest_folder, and removes the zip."""
#     os.makedirs(dest_folder, exist_ok=True)
#     zip_filename = url.split("/")[-1].split("?")[0]
#     zip_filepath = os.path.join(dest_folder, zip_filename)

#     try:
#         # Stream download
#         res = requests.get(url, headers=HEADERS, stream=True, timeout=60)
#         res.raise_for_status()
#         total_size = int(res.headers.get("content-length", 0))

#         with open(zip_filepath, "wb") as f, tqdm(
#             desc=f"  [Downloading] {zip_filename[:35]:<35}",
#             total=total_size,
#             unit="iB",
#             unit_scale=True,
#             unit_divisor=1024,
#             leave=False,
#         ) as bar:
#             for chunk in res.iter_content(chunk_size=1024 * 1024):  # 1MB buffer
#                 if chunk:
#                     f.write(chunk)
#                     bar.update(len(chunk))

#         # Extract ZIP contents
#         with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
#             for member in zip_ref.namelist():
#                 # Avoid directory markers or hidden OS metadata files
#                 if member.endswith("/") or member.startswith("__MACOSX"):
#                     continue

#                 extracted_filename = os.path.basename(member)
#                 target_csv_path = os.path.join(dest_folder, extracted_filename)

#                 # Extract CSV directly into the sample folder
#                 with zip_ref.open(member) as source, open(target_csv_path, "wb") as target:
#                     shutil.copyfileobj(source, target)

#                 print(f"  Extracted: {extracted_filename}")

#         # Delete the zip archive immediately
#         if os.path.exists(zip_filepath):
#             os.remove(zip_filepath)

#     except Exception as e:
#         print(f"  [Error processing {zip_filename}]: {e}")
#         if os.path.exists(zip_filepath):
#             os.remove(zip_filepath)


# def run_pipeline():
#     print(f"Starting SynPUF ingestion into: {ROOT_DIR}\n")

#     for sample_num in SAMPLE_RANGE:
#         page_url = BASE_URL.format(sample_num)
#         sample_folder_name = f"sample {sample_num}"
#         sample_dir = os.path.join(ROOT_DIR, sample_folder_name)

#         print(f"==================================================")
#         print(f"Processing {sample_folder_name.upper()}...")
#         print(f"==================================================")

#         try:
#             resp = requests.get(page_url, headers=HEADERS, timeout=30)
#             resp.raise_for_status()
#         except Exception as e:
#             print(f"[Error] Could not fetch webpage for Sample {sample_num}: {e}")
#             continue

#         soup = BeautifulSoup(resp.text, "html.parser")
#         links = soup.find_all("a", href=True)

#         # Collect unique target URLs
#         target_urls = set()
#         for a in links:
#             href = a["href"].strip()
#             if is_target_file(href):
#                 full_url = urljoin("https://www.cms.gov", href)
#                 target_urls.add(full_url)

#         print(f"Found {len(target_urls)} matching files for {sample_folder_name}.")

#         for file_url in sorted(target_urls):
#             download_and_extract(file_url, sample_dir)

#     print("\nDownload and extraction complete!")


# if __name__ == "__main__":
#     run_pipeline()

In [2]:
import os
import re
import shutil
from urllib.parse import urljoin
import zipfile
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
BASE_URL = (
    "https://www.cms.gov/data-research/statistics-trends-and-reports/"
    "medicare-claims-synthetic-public-use-files/"
    "cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf/de10-sample-{}"
)

# Root output directory in Kaggle
ROOT_DIR = "/kaggle/working/synpuf_data"

# Choose sample folders to process (e.g., range(1, 4) for testing or range(1, 21) for all)
SAMPLE_RANGE = range(1, 21)

# Target file patterns: Inpatient, Outpatient, Beneficiary (08-10), Prescription Drug Events, & Carrier Claims
TARGET_PATTERNS = [
    re.compile(r"inpatient_claims", re.IGNORECASE),
    re.compile(r"outpatient_claims", re.IGNORECASE),
    re.compile(r"(2008|2009|2010)_beneficiary_summary", re.IGNORECASE),
    re.compile(r"prescription_drug_events", re.IGNORECASE),
    #re.compile(r"carrier_claims", re.IGNORECASE),
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}


def is_target_file(url: str) -> bool:
    """Checks if the URL matches any of the required SynPUF file types and is a ZIP."""
    url_lower = url.lower()
    if not url_lower.endswith(".zip"):
        return False
    return any(pattern.search(url_lower) for pattern in TARGET_PATTERNS)


def download_and_extract(url: str, dest_folder: str):
    """Downloads a zip archive, extracts CSVs into dest_folder, and removes the zip."""
    os.makedirs(dest_folder, exist_ok=True)
    zip_filename = url.split("/")[-1].split("?")[0]
    zip_filepath = os.path.join(dest_folder, zip_filename)

    try:
        # Stream download
        res = requests.get(url, headers=HEADERS, stream=True, timeout=60)
        res.raise_for_status()
        total_size = int(res.headers.get("content-length", 0))

        with open(zip_filepath, "wb") as f, tqdm(
            desc=f"  [Downloading] {zip_filename[:38]:<38}",
            total=total_size,
            unit="iB",
            unit_scale=True,
            unit_divisor=1024,
            leave=False,
        ) as bar:
            for chunk in res.iter_content(chunk_size=1024 * 1024):  # 1MB buffer
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

        # Extract ZIP contents
        with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
            for member in zip_ref.namelist():
                if member.endswith("/") or member.startswith("__MACOSX"):
                    continue

                extracted_filename = os.path.basename(member)
                target_csv_path = os.path.join(dest_folder, extracted_filename)

                # Extract CSV directly into sample folder
                with zip_ref.open(member) as source, open(target_csv_path, "wb") as target:
                    shutil.copyfileobj(source, target)

                print(f"  Extracted: {extracted_filename}")

        # Delete the zip archive immediately to conserve disk space
        if os.path.exists(zip_filepath):
            os.remove(zip_filepath)

    except Exception as e:
        print(f"  [Error processing {zip_filename}]: {e}")
        if os.path.exists(zip_filepath):
            os.remove(zip_filepath)


def run_pipeline():
    print(f"Starting complete SynPUF ingestion into: {ROOT_DIR}\n")

    for sample_num in SAMPLE_RANGE:
        page_url = BASE_URL.format(sample_num)
        sample_folder_name = f"sample {sample_num}"
        sample_dir = os.path.join(ROOT_DIR, sample_folder_name)

        print(f"==================================================")
        print(f"Processing {sample_folder_name.upper()}...")
        print(f"==================================================")

        try:
            resp = requests.get(page_url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
        except Exception as e:
            print(f"[Error] Could not fetch webpage for Sample {sample_num}: {e}")
            continue

        soup = BeautifulSoup(resp.text, "html.parser")
        links = soup.find_all("a", href=True)

        # Collect unique target URLs
        target_urls = set()
        for a in links:
            href = a["href"].strip()
            if is_target_file(href):
                full_url = urljoin("https://www.cms.gov", href)
                target_urls.add(full_url)

        print(f"Found {len(target_urls)} matching files for {sample_folder_name}.")

        for file_url in sorted(target_urls):
            download_and_extract(file_url, sample_dir)

    print("\nDownload and extraction complete!")


if __name__ == "__main__":
    run_pipeline()

Starting complete SynPUF ingestion into: /kaggle/working/synpuf_data

Processing SAMPLE 1...
Found 6 matching files for sample 1.


  Extracted: DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv


  Extracted: DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv


  Extracted: DE1_0_2008_to_2010_Inpatient_Claims_Sample_1.csv


  Extracted: DE1_0_2008_to_2010_Outpatient_Claims_Sample_1.csv


...

  Extracted: DE1_0_2008_to_2010_Outpatient_Claims_Sample_20.csv


  Extracted: DE1_0_2009_Beneficiary_Summary_File_Sample_20.csv


  Extracted: DE1_0_2010_Beneficiary_Summary_File_Sample_20.csv


  Extracted: DE1_0_2008_Beneficiary_Summary_File_Sample_20.csv


  Extracted: DE1_0_2008_to_2010_Inpatient_Claims_Sample_20.csv

Download and extraction complete!
